In [ ]:

from pathlib import Path
import sys


repo_root = Path.cwd().resolve()
while not (repo_root / "src").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
for candidate in (repo_root, repo_root / "src"):
    candidate_str = str(candidate)
    if candidate.exists() and candidate_str not in sys.path:
        sys.path.insert(0, candidate_str)

In [ ]:
from pathlib import Path
import sys
repo_root = Path.cwd().resolve()
while not (repo_root / "src").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
for candidate in (repo_root, repo_root / "src"):
    candidate_str = str(candidate)
    if candidate.exists() and candidate_str not in sys.path:
        sys.path.insert(0, candidate_str)

In [ ]:
from src.drive_service.auth_service import load_creds
from src.drive_service.drive_client import get_drive_service
from src.drive_service.logging_utils import get_logger
from src.pipeline_paths import build_pipelines_paths
from src.scan_directory.cli import _get_root_name

root = "1hxSSsFQNMo63L_dDEbyNtVTa1I7IUjQC"
paths = build_pipelines_paths(root)
creds = load_creds()
drive = get_drive_service(creds)
root_prefix = _get_root_name(drive, root)
logger = get_logger()


In [ ]:
from drive_service.drive_client import list_children


sub_folders = list_children(drive, root)
print( "Sotto Cartelle", len(sub_folders))
sub_folders

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

from scan_directory.config import exclude_terms_normalized
from scan_directory.scan_service import build_folder_report

workers= 8 

reports = []
total_included = 0
with ThreadPoolExecutor(max_workers=workers) as pool:
        futures = [
            pool.submit(
                build_folder_report,
                creds,
                emp,
                exclude_terms_normalized,
                root_prefix=root_prefix,
            )
            for emp in sub_folders
        ]
        for i, f in enumerate(as_completed(futures), 1):
            report = f.result()
            reports.append(report)
            total_included += report["counts"]["included"]
            logger.info(
                "Progress %s/%s employees, %s files",
                i,
                len(futures),
                total_included,
            )

In [ ]:
filtered = [file  for r in reports for file in r["filtered"]]
filtered

In [ ]:
from drive_service.schema import IndexFile


included_map: dict[str, IndexFile] = {}
filtered_map: dict[str, IndexFile] = {}

for report in reports:
    for item in report["included"]:
        file_id = item.get("file_id")
        if not file_id:
            continue
        if file_id in included_map:
            logger.warning("Duplicate file_id in included map: %s (last one wins)", file_id)
        included_map[file_id] = IndexFile(**item)
    for item in report["filtered"]:
        file_id = item.get("file_id")
        if not file_id:
            continue
        if file_id in filtered_map:
            logger.warning("Duplicate file_id in filtered map: %s (last one wins)", file_id)
        filtered_map[file_id] = IndexFile(**item)

In [ ]:
from drive_service.index.map_index import MapIndex

included_path =  paths.scan_output / "included_index.json"
filtered_path =  paths.scan_output / "filtered_index.json" 
included_index = MapIndex.generate_index(root, len(sub_folders), included_map)
filtered_index = MapIndex.generate_index(root, len(sub_folders), filtered_map)
included_index.save_index(included_path)
filtered_index.save_index(filtered_path)